# Simple Video Stabilization with Template Matching — OpenCV

This notebook converts the MATLAB **Simple Video Stabilization with Template Matching** example into a Python/OpenCV implementation.

### What we do

1. Read the shaky video.
2. Take the first frame as the reference.
3. Select a small template from the reference frame.
4. Find that template in every frame using `cv2.matchTemplate()`.
5. Calculate the camera motion (offset).
6. Translate each frame in the opposite direction to stabilize it.
7. Track the maximum/minimum offsets.
8. Crop the stabilized video to remove invalid borders.
9. Compare the original, stabilized, and cropped videos.

> **Important:** This is the simple version. The template and search area are not updated during the video. That makes the logic easy to understand, but it can be less robust when the object changes appearance or leaves the frame.


## 0. Install / import

The main library is OpenCV (`cv2`). We also use NumPy for image operations and Matplotlib for displaying frames inside the notebook.

If OpenCV is already installed, you can skip the installation cell.


In [1]:
# If needed, run this once:
# !pip install opencv-python matplotlib numpy

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("OpenCV:", cv2.__version__)


OpenCV: 5.0.0


## 1. Paths

Put the input video in the same folder as this notebook, or change `VIDEO_PATH` to the correct location.

The original MATLAB example uses `ShakyStreet.avi`.


In [ ]:
VIDEO_PATH = Path("ShakyStreet.avi")
STABILIZED_PATH = Path("simpleStabilizedVideo.mp4")
CROPPED_PATH = Path("simpleStabilizedVideoCropped.mp4")

if not VIDEO_PATH.exists():
    raise FileNotFoundError(
        f"Video not found: {VIDEO_PATH.resolve()}\n"
        "Put the video in the notebook folder or change VIDEO_PATH."
    )

print("Input:", VIDEO_PATH.resolve())


## 2. Open the video

`cv2.VideoCapture` is the OpenCV equivalent of MATLAB's `VideoReader`.

We first read the first frame because it will be our reference frame.


In [ ]:
cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    raise RuntimeError("Could not open the video.")

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

ok, frame1 = cap.read()

if not ok:
    cap.release()
    raise RuntimeError("Could not read the first frame.")

print(f"FPS: {fps:.2f}")
print(f"Frames: {frame_count}")
print(f"Resolution: {width} x {height}")


In [ ]:
# OpenCV reads frames as BGR.
# Matplotlib expects RGB, so convert only for visualization.

frame1_rgb = cv2.cvtColor(frame1, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 6))
plt.imshow(frame1_rgb)
plt.axis("off")
plt.title("First frame")
plt.show()


## 3. Create the template

The template should be a small, distinctive part of the image.

In the MATLAB example, the reference point is approximately:

```text
[x, y] = [60, 140]
```

and the template size is `20 × 20`.

OpenCV uses the same image coordinate convention for pixel locations:

- `x` → column
- `y` → row

So we crop a `20 × 20` region centered around `(60, 140)`.


In [ ]:
# Reference point of the template
template_origin = np.array([60, 140], dtype=np.float64)

template_w = 20
template_h = 20

x_center, y_center = template_origin.astype(int)

xmin = x_center - template_w // 2
ymin = y_center - template_h // 2

template = frame1[ymin:ymin + template_h, xmin:xmin + template_w]

if template.size == 0:
    raise ValueError("Template is outside the image.")

print("Template shape:", template.shape)
print("Template top-left:", (xmin, ymin))


In [ ]:
# Display the template
plt.figure(figsize=(4, 4))
plt.imshow(cv2.cvtColor(template, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Template")
plt.show()


In [ ]:
# Show the reference frame with the template location

frame1_with_box = frame1.copy()

cv2.rectangle(
    frame1_with_box,
    (xmin, ymin),
    (xmin + template_w, ymin + template_h),
    (0, 255, 0),
    2
)

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(frame1_with_box, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Reference frame + template")
plt.show()


### Why this template?

A good template should contain distinctive visual information:

- clear edges
- corners
- strong intensity changes
- a pattern that is unlikely to appear elsewhere

A flat region such as a large area of sky or wall is usually a poor template because many locations may look similar.


## 4. Template matching

MATLAB uses `vision.TemplateMatcher`.

In OpenCV, the equivalent operation is:

```python
cv2.matchTemplate()
```

Here we use:

```python
cv2.TM_SQDIFF_NORMED
```

because the MATLAB example is based on a squared-difference style matching idea.

For `TM_SQDIFF_NORMED`, **smaller is better**, so we take `minLoc`.

`matchTemplate()` returns the **top-left corner** of the matched template. Since our reference point represents the template center, we convert the detected top-left point to the template center before calculating the offset.


In [ ]:
def find_template_center(frame_gray, template_gray):
    """Find the template and return its center point and matching score."""

    result = cv2.matchTemplate(
        frame_gray,
        template_gray,
        cv2.TM_SQDIFF_NORMED
    )

    min_val, _, min_loc, _ = cv2.minMaxLoc(result)

    top_left = np.array(min_loc, dtype=np.float64)

    template_h, template_w = template_gray.shape[:2]
    center = top_left + np.array(
        [template_w / 2, template_h / 2],
        dtype=np.float64
    )

    return center, min_val


## 5. Initialize offset tracking

The offsets tell us how much the camera/object moved relative to the original template position.

We keep the minimum and maximum values so that later we can calculate the common valid area shared by all stabilized frames.


In [ ]:
x_min_offset = 0.0
x_max_offset = 0.0
y_min_offset = 0.0
y_max_offset = 0.0

template_gray = cv2.cvtColor(template, cv2.COLOR_BGR2GRAY)

print("Template origin:", template_origin)


## 6. Stabilize the video

For every frame:

```text
frame
  ↓
convert to grayscale
  ↓
find template
  ↓
calculate offset
  ↓
move frame by -offset
  ↓
write stabilized frame
```

The key idea is:

```text
detected position - reference position = camera motion
```

Then:

```text
stabilized frame = frame shifted by -camera motion
```

So if the camera moves right by `+10` pixels, we move the frame left by `-10` pixels.


In [ ]:
# Reset the video reader to the beginning
cap.release()
cap = cv2.VideoCapture(str(VIDEO_PATH))

# MP4 writer
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(
    str(STABILIZED_PATH),
    fourcc,
    fps,
    (width, height)
)

if not writer.isOpened():
    cap.release()
    raise RuntimeError("Could not create the output video.")

# Recreate grayscale template once
template_gray = cv2.cvtColor(template, cv2.COLOR_BGR2GRAY)

offsets = []
match_scores = []

for frame_idx in range(frame_count):

    ok, curr_frame = cap.read()

    if not ok:
        break

    # 1. Convert current frame to grayscale
    curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)

    # 2. Find template
    detected_center, score = find_template_center(
        curr_gray,
        template_gray
    )

    # 3. Calculate motion relative to the reference position
    offset = detected_center - template_origin

    offsets.append(offset)
    match_scores.append(score)

    # 4. Translate in the opposite direction
    tx = -float(offset[0])
    ty = -float(offset[1])

    translation_matrix = np.float32([
        [1, 0, tx],
        [0, 1, ty]
    ])

    stabilized_frame = cv2.warpAffine(
        curr_frame,
        translation_matrix,
        (width, height)
    )

    # 5. Write stabilized frame
    writer.write(stabilized_frame)

    # 6. Track min/max offsets
    x_min_offset = min(x_min_offset, offset[0])
    x_max_offset = max(x_max_offset, offset[0])
    y_min_offset = min(y_min_offset, offset[1])
    y_max_offset = max(y_max_offset, offset[1])

cap.release()
writer.release()

print("Stabilization complete.")
print("Saved:", STABILIZED_PATH.resolve())


## 7. Inspect the detected motion

The detected template positions can also be viewed as motion curves.

This is useful for understanding what the stabilizer is actually doing.


In [ ]:
offsets = np.asarray(offsets)

plt.figure(figsize=(12, 5))
plt.plot(offsets[:, 0], label="X offset")
plt.plot(offsets[:, 1], label="Y offset")
plt.axhline(0, linewidth=1)
plt.xlabel("Frame")
plt.ylabel("Offset (pixels)")
plt.title("Estimated camera motion")
plt.legend()
plt.grid(True)
plt.show()

print("X offset range:", x_min_offset, "to", x_max_offset)
print("Y offset range:", y_min_offset, "to", y_max_offset)


## 8. Crop the invalid borders

After translation, some pixels near the edges may be empty because the frame was shifted.

For example:

```text
original frame
┌─────────────────────┐
│                     │
│       content       │
│                     │
└─────────────────────┘

after stabilization
     ┌─────────────────────┐
     │       content       │
     └─────────────────────┘
     ↑ shifted image

The outer regions are not valid in every frame.
```

We therefore calculate the region that is visible in **all** stabilized frames.


In [ ]:
# Convert offsets to integer crop boundaries

x_min = int(np.floor(x_min_offset))
x_max = int(np.ceil(x_max_offset))
y_min = int(np.floor(y_min_offset))
y_max = int(np.ceil(y_max_offset))

# Valid region shared by all frames
crop_x1 = abs(x_min)
crop_y1 = abs(y_min)

crop_x2 = width - x_max
crop_y2 = height - y_max

# Safety checks
crop_x1 = max(0, crop_x1)
crop_y1 = max(0, crop_y1)
crop_x2 = min(width, crop_x2)
crop_y2 = min(height, crop_y2)

if crop_x2 <= crop_x1 or crop_y2 <= crop_y1:
    raise ValueError("Calculated crop region is invalid.")

print("Crop:")
print("x:", crop_x1, "→", crop_x2)
print("y:", crop_y1, "→", crop_y2)
print("Cropped size:", crop_x2 - crop_x1, "x", crop_y2 - crop_y1)


## 9. Create the cropped stabilized video

Now we read the stabilized video and keep only the common valid region.


In [ ]:
stable_cap = cv2.VideoCapture(str(STABILIZED_PATH))

if not stable_cap.isOpened():
    raise RuntimeError("Could not open stabilized video.")

cropped_width = crop_x2 - crop_x1
cropped_height = crop_y2 - crop_y1

cropped_writer = cv2.VideoWriter(
    str(CROPPED_PATH),
    fourcc,
    fps,
    (cropped_width, cropped_height)
)

if not cropped_writer.isOpened():
    stable_cap.release()
    raise RuntimeError("Could not create cropped output video.")

while True:
    ok, frame = stable_cap.read()

    if not ok:
        break

    cropped_frame = frame[
        crop_y1:crop_y2,
        crop_x1:crop_x2
    ]

    cropped_writer.write(cropped_frame)

stable_cap.release()
cropped_writer.release()

print("Cropped stabilization complete.")
print("Saved:", CROPPED_PATH.resolve())


## 10. Compare the videos

Jupyter does not have MATLAB's `implay()`, so we use OpenCV/Matplotlib to inspect representative frames.

We compare:

1. Original shaky frame
2. Stabilized frame
3. Cropped stabilized frame


In [ ]:
def read_frame_at(video_path, frame_number):
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
    ok, frame = cap.read()
    cap.release()

    if not ok:
        return None

    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


sample_frame = min(30, frame_count - 1)

original_sample = read_frame_at(VIDEO_PATH, sample_frame)
stable_sample = read_frame_at(STABILIZED_PATH, sample_frame)
cropped_sample = read_frame_at(CROPPED_PATH, sample_frame)

plt.figure(figsize=(12, 5))
plt.imshow(original_sample)
plt.axis("off")
plt.title("Original / shaky")
plt.show()

plt.figure(figsize=(12, 5))
plt.imshow(stable_sample)
plt.axis("off")
plt.title("Stabilized")
plt.show()

plt.figure(figsize=(12, 5))
plt.imshow(cropped_sample)
plt.axis("off")
plt.title("Stabilized + cropped")
plt.show()


# Final concept

The complete stabilization pipeline is:

```text
Reference frame
      │
      ▼
Select distinctive template
      │
      ▼
Template matching on each frame
      │
      ▼
Detected template position
      │
      ▼
Position − reference position
      │
      ▼
Estimated camera motion
      │
      ▼
Translate frame by negative motion
      │
      ▼
Stabilized video
      │
      ▼
Crop invalid borders
      │
      ▼
Final stabilized video
```

### MATLAB → OpenCV mapping

| MATLAB | OpenCV |
|---|---|
| `VideoReader` | `cv2.VideoCapture` |
| `VideoWriter` | `cv2.VideoWriter` |
| `read()` | `cap.read()` |
| `im2gray()` | `cv2.cvtColor(..., cv2.COLOR_BGR2GRAY)` |
| `imcrop()` | NumPy array slicing |
| `vision.TemplateMatcher` | `cv2.matchTemplate()` |
| `imtranslate()` | `cv2.warpAffine()` |
| `insertShape()` | `cv2.rectangle()` |
| `implay()` | OpenCV/Matplotlib video inspection |

### Main limitation

This implementation keeps **one fixed template** for the whole video.

That is intentionally simple and matches the learning goal of the MATLAB example. A more robust version can:

- restrict matching to an ROI,
- update the ROI after each successful match,
- update the template,
- reject bad matches,
- use feature-based tracking when appearance changes.

That is the natural next step after understanding this basic pipeline.
